# 独立放大版 RMP 语言模型（~138M，Colab Pro A100 一键单文件版）

- **架构**：纯自研 `WaveNet`（12 波 × 8 模块 = 96 模块，每 token 激活 2 个，25% 稀疏激活 + Gated DeltaNet + 多尺度语境路由 `wave_ctx`）
- **参数**：~138.86M 总参数 / ~61.04M 激活参数（GPT-2 BPE 50304 词表，权重共享）
- **数据**：英文 `FineWeb-Edu (sample-10BT)` 流式加载，不占硬盘
- **免上传依赖文件**：代码已内嵌在 Cell 3，打开本 Notebook 选 **A100 GPU** 依次运行即可！
- **防断连**：每 500 步原子存盘到 Google Drive（`MyDrive/rmp_138m/`），断连重跑自动续训。


In [ ]:
# 1. 检查显卡 & 挂载 Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 安装依赖（A100 Linux 上安装 fla 融合内核）
!pip install -q tiktoken datasets einops flash-linear-attention
import torch, fla
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| BF16:', torch.cuda.is_bf16_supported(), '| fla OK')

In [ ]:
# 3. 自动释放完整代码到 /content/rmp（已内嵌全部代码，无需手动上传其他文件）
import base64, zlib
from pathlib import Path
BUNDLE = {"dspn_text.py": "...", "pool.py": "eNrdfetvG0eS+PcA+R/6srjzjEzRpJ3NbZxwsQd7oxwQ64w4QT4IOmJEjkSeKJImh6bs/e1BfugtWd7EthxLfsYPxbEleeOHLMk2kPtPspoh+Sn/wq+quqen50E9snfA4YzEJme6q6vr1VXV1c333nuv+XKlcfmps3THvjHvPL39y+ZEc/lNc/lO4/FjdvAQa/wAj2ac59PNlSuNy0v2yLetkclfNif/NnLu3XfefefzJHQYKmXN1L6qZVjmPnj1y+b1bra19qhxcbV5Z4YdPXG8m3H40NYZvdsaud46+w3LsP3MfvtDc3nVfjvGTuC3pyOt80vs2GfHCcqMszLnTM86M5P24qp9YwQGZoMI2Fl45lxdhQbvvuMsPoLHHHrj+gWW+dvI2RPMvnjLuf1y6+0NZ+Ysjjn5gzM31xqfac0vMYOlWBJa2WPf2qP3WYLPpfnmG3t83bm20rz7gwTIctDuDAO49tRt58Z5++sZmvbnCTnpoUIZpjxjr7xqjD7kc99aG8HJL922V8aw6xsgxIpze803eWXqcqKt868by/N86vbcI2dqyiMIc2bvOhOXWDfjVADAW2vf2MsvBB84F2F059b95spdVoFZFkuVIe2rdIX99JJVhqr0tabrAB1nOP29PTHmXFllmVJVq8SYkc7rgFZfOs/suav25Cwb/GVzASj88qV9aQJm3xqfZdVSvzVkDGvQhx1gljlU1qH1I3txiXPGHhu1X38NFLBnrsI4W2uz9qUf7LNzjdfLMP2jplk+YZqDzJm/3Xwzak9978zesu89tW+MN+/cAeR5SwDYWL4DODZXz7qSxiUAkYRmjR9ubq1P2k8ebL0ieVVYhgSnpvalv9ijLzg18OGNcXtks/FwA3BzntznuDlPz9krm0TwBT40Th+JODPZfHar+fo1UKLx+IE997K5ep7w2FqfaY6MiimXC0bVymdSX1RqJsz+cnP5SWPjPIfqzL9wxXQB2jbW33J8t9YuNlcubG2uQgN7bKOx8Q3MoDX2tfJ1hqte/BBrTp2TBLh4y7453br2wll+bo8ugVjwCXOhscdmt15dcy5+37hNAv/ku8adZWfk4dbmNXt0gs+2sXgWW04tAEZihH9WR3jvvffefSc/VC5VLDZkWDn5xSpVMrkY/ydeLDKjyopF73u8v1bMWPlS0Sjgq0/efae/UhoSr2tWvlCNZ3JmZrBcyhctJmB6T0TzbLVcTFvmsGzxmXHarKBov/tO1uxn5pCRrmaMojYcY2YCxDXGKmbVtFLdpaIZY0dSH7yvH373HQZ/YCL22gMQV2AosN6ZGQcxdzb+Yk8t2a9+BElgZtoC/RhIWz+9NNN/sjqTfwbJ15I/T3xt6D+9HE5bQLEBamMANbc2rzdXbjTfvGpcmbHvXXBNBggJO8JAb0GPyLZdh79BAcBKNu+s2uNzjVvft649R2UGNs0/29q8zUyiNCJpsI9TLAGa8KKx8AyMAnJt/rZ9ccye+yvvjoIAEnTxfvPtZXvhpv36G1BKtFNgtYY5CsOHmdYXY1/E2FFQ6wUzwb8f1dGiUS/vPQjoRWcR9OIiqsD0LFpLRIfYjhjl+wVS8YQgJP4Z7gcqDMf7CyXD0nTvecW0apUivI7B/z2HY6wz2cvfigGpWzVnlE3+uDAAT7hQ9NcKBY0Q02MkavFCaUAz4EvWPJXPmKnhOP+gS8yI1ywPkleyGLJcQZEgFwbiQ0Z10Mym+/MAndoDUodgNjrb+c9vmDlc1kAADkHznycnkMnXJcvtt6Bbk86Te8ABsrxTW2vTztVxjkOpZlVBKAGJnl4UTj+x+ksVZiVYvsgqRnHA1BJEniO6SuMMEguJaCUOQ9v97EivC+Qj9hnNLvA2Uxuq1oa0JLynzhlO6h6XCfjHquQlyYFiVS0TY/Bf1jpdNlP8cV+pVAhTPQ49Cyqzj0k4SCXtM0QG/kM+9LJORt+5Hh7u1X18+E8AFWN8Lvs688X+fbquAD4NgLUkgDB01uEOkS/i1Pb15f8j1vcf2c7f9+Wz+2LsGAhaBlcoD5HP9J54PO7i0cFMFw+FCMiW00I+PyJWxY1y2SxmtdMCDyHJHGwG8OT8TII4mtz+ZMDOV1n6j8Nls2IdKVVMjYyTYm8+T+CKTIv8dSRWEr9y/8f1fGaiFnNYMvkyjk4A2JOqWeiP18EWCLXtxtWa+zhba+tkebgroioumsd0Ol/MW+m0hhCgX4x9CXwF7YqxXD6bNcFen4yxPtMy0rB+q8JXrcGsND0uISS375uSQDwYWbPAUf8KOtOHPvfDVwfFh0KpYqQN9UufggTvjE5LMX7cqBhDpgVYcZ6A3mSLGkwpC9yXSOnggJDtqJ6sWBq9UiVLYBEN8YxZAaenW4IK9vvq4K76AUahrnXoiSKIcuMyB5xEwRdQv4pK/HyxLFULRa8HG/SisOpCxxKKLMOo0PqT+IBZqGkBZbHysWI+B8piFXOgLQDYZUASlUaQQ8E2B6AychBPryTAYgz+R/WzUP345OJWSQMs4mRD9Fi4Ty5WzFGfInaCplIM9MDYOdWIhNaVXBj2UDY2lD2JsIdOImwB+biuKulxMGiknBqw71gpWyuYipqK1bMml80vdVzf16btpyICkM449+LBi2zhSoweIm/AcrJzEvmPEYL65CQ+AbevMfXCGTm7s5oORmvbyUrqEGiOGlkpKpiMwyqC3ncqEYcx8bEBHxPic18qaXYe2lbJg2LbLag5KP7FgcVHHEd8pIGUz30o7DQH3p43Fa2oQWCYDBhP6MM51L1bQ4MOAA7AUikmiMHMQtX0meSdTF5QxSvbGZsvkQN+C/NlSNUr5kC+Cj3TfbX+fgCwz8hmK8CmT+IYahmF/BkzaMEAKOCYH0qBgu8Iri9vVPe5eiBMz869YKnNpsFblj25t9Udg5FBbmBOYSAikAGKUCjjOUWtuxd4UIPeLcVV4EXyUKl1fU7EO1FhDvi1zeU37BMD2BQKcCZV40gCSUwVStHnOiSqAAuzIIUornTzOqhwK7CMuyBrIKMQe1Ks4INK5tTj1idxiJPTPFCOMa0mnCpwHWK6zv7gSo5koaST0H9grwccY+QUDCG6oXDEMwXAAPTQwv9LWkUY0h1cU8UZWKDYfMVZfAwcSRtZo2xhtgN441x+xTlE2YNZiDaaD85CLOLMP4TIV1lyssPodAF2rosplweQN0SrPKi5tgDmCJTOAm2rPpGRixaJZbqQHzQRot+5JPCHDupxiNssFFGtEwwUjE+CqFAK1BvZ08a994bEvzuoqeL3eQ3rEis3TeGb5QHPnPl9VAIOcQ9O1/xQp1nDOPgYqd+8c7/1+tLW61nnyjPw58gPCwlmhaxijNVVEQTXv25Usp4Qkrz6A9dIuRSLE8ynpoZQNJ47EKcHGACScyHhfqLy+VYM0JXiAAMbJMiTr6YHKmglikZfwcwCbfCdagnC5O+Pc2lDmQWTmsmBBAvKEkYRIwdMdgAmrN9nYqxYdeeBWq3VVKcAfXxATEi/u0x9qfvpyEcnToEH7WUCkRyYylEThLQmY1Jq7pofmTp6JNy7ESoZKWEKwzXo0sFyINzgnRzk1EergAYUMOSvz7R7XaxGLIzkO9bb2zw/dYJUEOzn3f8gkjMlYrS72pMHwrnIRZILkyp5maLFGUKIY7AKCxXb+x8wWrDieHCHTKOYrnhhpM/LBKeueHKfEGxadgENsJXGUFkbyhdT3BkO8KJWzpIdA4R/zxJSGCNaKua3VD6d1lSLL9+5RpCcF2Adgu9gmsC70wMSuYDjoov+NCDDkcfZ6b4pBEMTMLYIj7uCwptyQ+BqfqCo8cWaiz0ggEOEFm53sY8P1QppLRH/8EOdA0VnEMBRJ9UkocxUaUiV6+gaU76zMfYcxiyUBlg3rNoYPdnLM/bokpuhncQk0pMNsVuwcnnrzTQmYN/+xVm8xSGwj1ljYdm5PY46mDyQFMky6QXjn7Kr8y767jzldIh8ShBPUsk53KmVYWJllbZmZxJWGcwh6VzhpOeGzzgFdV+4Stzi8ADcxwzcaElrKUpxZL6mh5NeYHLBLAFSMXYIxIF/RaD4XaV2pjRUzhfMdNmoRFDdU3x/O19A85VxysSgJjKe+fyg3G+grZsZyihw7tiXLthPb2Ku78e7wCPuHrI6gIN2iXi86+eJr5NinwXs5o93t97ccB7foR2IBfgKK0xzZcNZnrZfLjvPz7LhNOYl+kqWVTCLZmaQZ73Rq5t7BA0pJ4ZjuXHUpEx1Nh9+1xx5AOvm1tqks/xcblJQKvyJffd88+Wt5tojgJ5G9NIna2blNAE4C/DssVG+uQHN+b4FNqRVL81nQw0nH8LaDG/shXV7+TpubQAq44/sqaXm81V7YrwxOd5YPMu6mD0y61x7Yz+55ibDZ6ANLAuNhxvYBRqPLkHM5+bSvWk4L5/a6w8al2+1rryFgQYqpXraPAWopo6jBeKbPxA8suPIBOEb01aDYMjcCoxhT6w2FtZa53AwAI1ubGpff8Gw9jHUM3ChHz/GvDAybWZftVwBsYJXN8YxyzyxChPgHXcOK7ti7GgoIOJh5d4CyZh/ARSJC4l3jCmkSGBCtzIAWglQMPUDIm3i0wpIbsqNkKhDrWjlC/gqYw2ns2bGgN7xD/8Z8DFq6XIlX6ogjKjBFSGkACPGPLnxPQHIws2ihBOEYYMpRCoHSGVxaHKDMoNli3f7u0Lmrr3Ey127CJY9srpBOVFWfOGTiLlNxTxwW0PpJnuIxrJdcDWSFHUzPZKgAFF9673wQPxGcBfIvQ/3wVR2uFaitXGtuXzP3hyxL620Ri6BEDemHzd+mOYWgP3x2L+ATkmmYS+uUKgOdZR5e+WVvXyz9e0oGCN79D5EQAgDFplhZp+9zgEsqDihlm/O4a6LBEtMZqCI9uimvfwKQqPG5IQ9+nRr/YfG5lXQd/vpOYTEnO9GwEY1zr0CTZPiyZzLK87MWXv2ivN8Gd7hrtLEVbAGYBYwKuiineKJ+eadJQx5aZL2vev26jo0cPd0PHvipQwFeujOYEJe88hHqQ/5HuIjlGae+/BawiudGiIXyOEmRlArmnEwZQDNYoqIkSKKB95kU4y3U5vIt8FFzH2O3nwPXz4zOm19ZHDnQ77vRTTzVfD+LKOYgZjRfQEecyFfhQXVqpVhidM59i4ot5WOWf4ub3BYIM2KxUA2tQAeOtIBlKx/nzcjzqnW4kjjyeTW+kXWlfpT15/3+eIWScGcUc3ti4qCQvmWnGX0yVwLJnq0g7/9gBTc53OowAciwiGeWhjYLil1FEH6clJHQw5hMMEWWgHgAZlBaRC3z+y74kfSGInytnk0vgYFU2k+tJHRewe8E1QCURqA0DcaCu2HdUeFNr9h4DnxFZu7Mvbdm42FKUqAUX7rEjgjS7jbc3WVN9vauAgrdvP5S9zDvboKHkLj/s3Gxpw9et4enaBw8y/2d6uU9/q/l0CsJ/zJINqoUbJAhVJxgKwTLqMk/tzbIQXnTQy+M9qNWZcudgAG2AlHlEnArx7IIknPARyy5OG9xauNjW+cmxeAYa1HM7j3+2IKHD5k8493cfmaf2avrYHFh+fc/5MLE/uveTnuv9ep1mKq9e0lljTfZ+T/TfL6i+bDC+CZwnq39WoykH3IWzkWiNgPh/0ew6M0BIhDpXxW80KIvrShh7sAYnyH9YDYZk3QPqskU0cH0DAUOsvAFT1E8/0IuEaSNjk7GUIGcOEWPsxE4O0KxIBmeBglfRvCwIfv11vzf5X1YmotDPu3g9LLxsX/yrfcv0R/G6ua3GQ18Iu07pwMGTBrLULYCSNj5U+hkeQuu89nAB63vt/gHoFEgTtPPh8fV/bJWeFWeTEX3zqy345SjZyC0ORCa+Q6FoasPHF+vILVZ2RLAK3GjTuNxdvO6trWq1GeH/c5CICsfw+/W9XfXFD1lJe4+Na9soOuoESJPDCqb4rVdZC8IgLhYpAIJL0wpcfniplCXG+he5SMVnM98KbnMG/d24tiEk+EgRHbdoAFcydgndT6MAFL7GhCibtgG+DDjqYuX01zRDH5lNt5f6e/P2hOYaH3LSLK6oEyd4nMhyxCo+qjrdff2psPQK7IZd1xTYAv5cComHINmdiQJ2CUrRo5A+Qz7sESNpffOBOXuCOLZWwk7s23r+2p20xDyZ97KXJ1am0gfBOlhzqulRCA37yEJOCL5sRYa3QWFGCn2Q6VThE3AmtWCEewsZisoJgdjbTQ1etba9OYhnh8B9QTSwAh3rj3zFkbBatiL5/f2niMdRmUBt52CyzwslY109VCPmMGd8h2pKWajsA9MTcXIfMVgaQEWgSRl0DbxOtueXUphS0/BjBD1QDkpE3jEc5uMHMDM87erY1RBm5pzqwwHmXxosqttSdI3WcbzZfnpT38N21QZxxLTAcS3pjloOiIdad+mzx4IJk4+H5YuAW2exbNHWSyTtqeHiiU+gzcGrEEPbB4ZuvNAvv82PETeYwBKqyxNNK4Oode3dx3YXFM9+eHzSxntdyYSGN2DrNN6kvNl+eLbhLM97n7Il6Afjhya4GXj6i7WjJM9PZeMuUa7SFi1ORLTdDYMVam3UcsSEsE1oS8tyjwvEXQ9kJsVsFVB0AEd0nyEMEg5I/dvDSa8HoPPOqlZSQfYcax/X5YBkKLAL7oFKMJgINuGp6b5QhokTTyJu4Wd9E3jWATLSKyuILLApFUyp0SOabBEp40ecVYqco3ndPZfMbdRekwKgNA8Y6OwTp+ikwiRXYPdvxvksbd7ahXLXXnrd1+elB2KYb9B5FiiBbfqpJcOuqO8VXF3Us/qBbnZQIOTJ+XSzsq6xIFMhIHmQ9RQgjZPdQtxHetA/MMZiaowUUrP1Ar1ao8Cd1Gh3GnZPq+L+OJGS++0G+treNhhfW55oOxxsJVlojHuzupFLD5FgLCJ60bN3EdogUATBOv0G1sfoPJaPAOCAYXS8wfnX1rj86yI18e/RfWVTHKOWa/vmIvvHEWb7Gu41/+PPb1keNfgrcxw2OM4B4LEAv30gPLF+h/wATJRyQFFeBI+134SN1Tdyl9Qukjr5dEEDIJNhsruTNpTE+6G7mk9Onhk6ENcZgahNWwAsioC4nKT64Ir4MKuS7ytCAvSbEvXcRWlGDkaz6OVzTrEeRSypV7OtUi2q8q6bonxD31Xr8Y8hQpCaJo0kZnUtE6E5TpCIs3fDIm8AY8ZBm8UqoCdNSOxnRdkjOQCwOkBYn1tvZUTTkMn8SN8GG3NuUPRAOZdXCRiRQN39aqy83toUVKU5uiHHem7aAoAudtEPkEDg+6ZLMVUYZSLURJGh5+Wp6XkoabSqNLztPbzsRVrrvQD0QIC8HJlT/Qml/iZ2kgsONOE989ohISFEr1CE3d8gV4aE3UAHMmHIK68eSEDGrB6JU7Bxk/ByP38AScA4Ly+wep5p8N+h7QmSiR2KLENC+H2LU+oMAWtrERIaGPSnZ65VHceltGMae1YbRIjGJUWi30ys3ZgCibhdDAUSlcb2Bqi7nbHpKEOkEPwgxlJr3+GhZ2oSj1QL9gPZcePi/Bg+P0kDu0DD2xP7Le3zSDjhWMwssOyJt1G+rsn5j2nwgrAB7TfS54/Nzjm1HEbP6nePU/xqRfxaC9MUeT5pjnRZDYfoYRD7bhVkRfEZ7thXnKMY5SWRTs4fE8qtHznbDAwUXp2u6K95Q6HH/5XqBSD4DxSj3WQTMM0w0irUpk0d22xXbYJ7LcLr3d4hklKWJ8DvMA/5cKQBB9X8lVm1IcdxSPs4dDUoEHoJBh0WJjFMo5w2WcAHNAsF18FSHNoN6mpjFNxUcwUHBylLqNJzBLiqMglWjG+5nGhz2gDktEDEERCyoVZ/LCQSu02rov0xUKnivtaxiHI2sYQTmj1lH1dBqThSTN5y8b5179srkAvUQDnfHDbfaTeTqPOIG8Zpgvn/gWgnRxQpefguMZNVpQWc634eqszG2tPXKHzOqiMGRxpLExIUdvV5y/zcE1qu8yjUrR5fKvqqgMBD1+r2c3nqKgO/Wmzz2HD7pFgD0H/Ss0R1eioCQc9rIh4k+ubW1cbC284JTDtNrDaT9+qG5DYldAlH+eMrMRcyBDSVgnekU571CPOD3mr23sSbZp4O5TRGwb0qhk1/wVG64zujvaE+1k/NTGOZcwBVd28s3DjnkiwjGnYsRUuGY8mOJREgKRkavSgeI/Sr1JQ+VLG+JkNTcd4nl2XIg1/EoCpR7gAYORKVppo4CFA7UhtBxGfz/GBRQfYFZKpJ/8/yuVpl8AjBO0UvNXFOBHoCF54U1DlAdoniz4bbrXMsCyzBdp3M2EgelfqUqJXpfMnuiFCR/YUonOnlEMhtJAh6t4vKi1kTCfdOlq5slnwpX4j7Nciaa3jaP1kL8nALX338UUkIFuLo33CSobEY+Y5QbFYc61mZC6GMk5KQFbdKimRzuvbjJ5+yntFLao7pGQjOE4zDotE0tKCXrUIYePZFdeOY7OOk0tehxYZn/lMKKnOoq6pIf3YDl9XGYicV1MZTY55kJVnnh1/npE2qCN673DqL4BdzXWdsos9gjThVLGKAj321e2TnXh7r6mUU1TbZGo9Iva5XQB8o0EPP0voYODBrAxnRy51Y0lfLjtwFnOeSM79+K74LNoEY3YJNmVzAZ7qXQPbo1sy9cczuQM/qWBrTTQUILfqSaKhUPIlxktGqPh9sc4PGzixdqQWUD2H9Vj0XCkn4DwvJ6KfxB4ofun6x0QiayOEKtBAAbW8cPUw81PtGl9Iqpxrk676xYdJvCqkXqUrjJKpnMnyJjtT56IuwbSZxAywO9ARokTJlGslJshvHHOaxwZl+9kHou0/qAWQEDauY0+1Chjuo0MFOvI8wh+JDCndyLBM64K66tBlle3UyCvGFausbtx2dtoF2UTSdjT/UWtRussIIr/nOD/VKqWm3alf6F1qlpoB49rGSoZ1zFUse00jI+oqUPqNEZwYL39eO6hajGsGDW6PT+XBU2QFcQJ764WTRIihlyOKRxTlS1GNrtimkWge9Gtrd7TIsKx0CQa21OozgkkUEEfzqf8kaovtQPHgHX1hHyAw7VXaL86V4Nq/CuUeA8q3FaBIwlZcS8j8N2g4nlm9Ei9QuOfwiksKmKYe8Rv0xG7K26xBY+4RZJ6cQmPW7glc63xGWdtlJdiRMyLuOtGg94Bt+Gdj/8dVY7/1dtxN5JX/wv4hBVXHKhrq7zSehHSDIOrQU3CjhAFYRFs3mX5IEkEpinDJzijG4sAT26p+70r9Hgx8HNf73ywL6u3GUiuPts4Odw3Q2tUqvKDWQKPj9zA00UEWnVCU/f0lpLQWFjDQwQzr1qjs/b6AzxcwFMcah3Idg6nTF8EwkhfNXNkJNQOkHfzB4XDSXmlBj46QU/0cGYnGNQWLd8lIjIwT+jbxWUiawuSMOh24WIbzG5XKz7wItzfPXDqEA0aWKeYqN0Ak1mGpN7+jHIGj+ZVeU4ixvBc5Rf63rJtEPOCFnq5bRpbxMU4dptiBDPThtEdMn2HcMNFCZhQcZOrezzEyycrczAgQfnMoK8KhULr4HaS3I0I7G5gaj7q4C6NsLujub/qwK0mj/ICUhzj8KHePRzJrcJoOOVPsG42nStZ3tLmFplI+ACWjxhRCs3FVpZ2YvkjbsHy4gOZHXWurCo7NkZlwJSZjUFM0wMyPXL8XoWAYmsgMgPLl1u8Uu76BX6olp+xbXPumPi+52PGAllxwDhECd2rycZqztDmNF4s+Gyl8XCF3wryK84nUwU54hB5GIMbbXmkmBPAPZs8cd2+/9pXZoiX2PX3b3+TBAhNf387jOBVsA4LK3B5m6ito7bVfNwg4WsN2b+bs/m8MWk1dfHvp9ibt1vjc/bMVRQHWrn4pYHs+GkrVyoyrEG4vPLL5oxaMzS3Yr++wuvWxa2VOB32j+w4aTpdQHj5lnNlIrgbki1JgyF6BCuHyFKgBVFafZwKHizT2f8TYLxnvKu6kXhKmlpDTa7KYi61AjJQ1pU1zbJJ6WZPw+O4EYkXmyTVswwcJp5aUPPthUKpzq2jVj/V495pJ0EppsDoqwJQmGKSpi1bwxOBhCo64iiWJ3Dqpq0YVeyBKntQfCbUGUgxgPNQsRUnmvh7XrCLF7ZhNw9RntuSFhLG9lpGzCuYCssYuOFMI7t+Nx7K+Z2CRmkQWoCEABUIo9+rB1SJODSRf1BKV/la47eD7Pcpr+4Tho3cMVfWMaM4qMjJQLVUsTT/t06OTwdgKC0YClE1A65hvjggYlF1gStR4TTMCMamET4mVCIOafqOsdSBoKaGvTn5FdKGavH5phTvjZ+V+ohdXM2AWk5nzwFNXkKIQdgVvAA5cBcDNOD2Ut7FkIgnsaaI75q6K1VE/ZtYLbddLIMOXKn461wK4fJgPoEEgudp6d4GeZuCuJvBt4j0lIq9fEqhKxr4RQyU39UQbpUfPq3iZo3P+P7dtzIggHq+mAVTlgFM4QVtT3waY6dKGaMvdfC3HygXNLjb27SxzX4e+5pvT/888bXbQRdnnC28SXbr1bR96SL7lK5SdjfDrf1J9vPIA2bt/5RXneGtsFdf2QtvYOX8VGXkEeJgGRYVj4+EHR9JEk1eHhpjGEQl6OK6hAiVfsOO9FiY1sQDTpaHCS6+y3ekn/MFtADYgWtH3RLbI2hTPkX7e/gL/BuNET3D7538TS8e3/w0eNdFt2lFXnXhXUfuLI44V1fFJeXXL7ATrLH+kF+PjIscIWs/n8GL/pDg4iJu93g3Hrnw38t5jB8fm8TGztQV+/VTe2LVu/eCHoPy0NXoeOEGNWyubNjj694VF/TVmd50Lt73TpLnmHcGSFQs4PCkv+51nqSdM+pV6VSz6oP++SHo1t+X+so7g2avz9kT81j095V3BbHvBo+VOczK8PMvU1P8eg5Uf/8VwKLoOMcaG3OthRf2xA374bSgwtRtlRBu3cYEljgD5I0Xzbe3ENfxRwC8OTKD1yjOzbYeTATMS8QFFcfwFoZs6kO8LSKV/MA9nZxK/g6enKQn5lAfXVUhtSpwa0VElqA79f7vYmww9X6MdeHlDkcJQDmb+gCelPlAZXekg7+Tt2F491/s9lYL4EXiv+NuC1bGfdr67u+5wNtA0lhQ4ntQteCfob/3LoyqWU7ns6lD4WsxMKQ008gR8bWjI72nqzKOiUWk363vz+RqRfeqiXKphHttx5CsZIzCp3/FpGPKdz5n9y4MQh1TW17TQCts4MHtgrZdWPpOno1Gx05EEVX7ghiqlH2AJ5FoNebHfOmKMVCy66z7gDa434UCShWYBZCPH0X/41CfmUVfRCPZJlEPHGPm1A7EE9AVyStiaTegG+qL1838QA6JY2VJPl07bt972FyZbd5ZOgCfGhvPCOdpca6gGyw/uLH2ymZz/BlXe7z3KW9VWWNirHFuTb19KSOvFDhG6Ebcs7mXSwX4qo7wqpwk3Np/hkeZemC0mHfrkXqVTTmLaix1GPXXG8e9T4VUGP/yKXD4TpSUvBqFtJj+FpocvPciFXkFRkq5u8LTavkppmpv9AUqKe+joqPuB1VPFboKfQ1d6JKSn/Re33lqXBqenmvdveA8n5bnqXEdxWPVyWTwlpW5B87ipLj89t4zfogZeuLd5+CZHPzp5fv/Xvc5BQh/7TzeAsZPzC9O22/m4XPz7eLfRs7Ch8ZfxW9aiAJ4bvOCxy/pYUz58imeQBNPew6CR/8+nVYPn2v2l77xLlGheDkHFiFC4ODrZ/miaVQ0FDDuKtEoad8ooYDAvVpCdD4GKJ7072sW0xV+09hRvgEjDZLI9Zez8jnaLPEUAWHRaBdec5YN5H/7+6Km1t9X9uHCuyIyH0mzwUt9Nbe9sBrbNKBS48Ck0faIkVB9u7HEjc8ycNWJMlfacuUWjT6TIQSaRN+uYmR53hWG+FdQcitvndZ2cVuIr6OgAkeM29foaYpeLi0icmfb3QPORSXYy6gNu6sSBnSMflAhxv70Z9/xFBLTQqlaFR5Rrk51ToHIL1eXJbRdP70EYWBBb3Kh+XqFJ8e21u/Zc5ONsecy3uPHzVHnH83bc981X4yij7r42Fl+jnlLvON6lv/gTDAa7KIhEb0YQymXF+xwHdJjPtc/xmcYPGcKwUUOqP+pjmpkYrFHBU8XnsmXVVg+ldf18AmFTzFJ8EVkmT9IR7EWqC2zBlAIoiK0wH5BoUzREqx5abdyPZfVcnXagqQgBS9Z7QDOeMdg0FnoOqzVKbEU9a5XuRS1M7hDgfREYuI/+1mnhqh2AB66KFl3o1mQ1cCBWRFWUc8DlJkp8g2Q7Q59qrTMmqfcVJTnMYhUWuBIaE8hfBaU28MCRdPe2t0bToaGzZOsItSiznlyQFhs2eXu1XqPKAjPquc59Zg/baicOFV0q2oOCBqYbvGglSa3Av4t0y9e9Ped8t8uXaV8vUaXYqRUp9Lb6RFO5g71jcdcOh9TqmVBO84k8NbXhLz5ks8y0auZUXveHaYQLKrxRqfLEzTdm0+wdnhYnhXPF7UzeLELHuco+jeiaRU55UuR71wM7vsNnLkpnqhAh3hs1vlxCWJtGWCiaZl/yNvzeNAvEf19acyE+m414SdQhvlm/OEkLwWOKFjgP+tC912oaQyeYFKPGbbd1kcKDbtbT7DKaUAKtaNvq0mg6i97HsYNPORlMsDLZK82rIiYKD9Box6x9RhYH2WpeaT7gs5BOufbhEVMxI9EoO3x9K9tEbH4vR4XexJHCHk57JBKKT9JkVPlCH8oQoytQmmHWPuTJEIvwcaEjo7IuhD61z2frBglb/FJqG6ZqZg4zU/36GxV8Jx0yHrlYEbqNNOyWhltjKnOgmr53RK1UE06eYNUwN3jQgsezdv7Dcs8EaMkfcQPUNmvXtDdIt7vTrl5mQXeoPl2AbPJIN548wGePqdTHXhbxsaGc2HOnhhHRcYLsRZBqSFADBh1zBYk0YqmMdLG38tRCvZUMrinU/yl/9wMnIq5GKQ4pMDdgAI6vxBQ6L94FiwjsBL+q4G8yyrgzceR3gPtVqXYUL7oeryw5HN8wPmBz1YioloF+J3OcFlSf5AJgUXUD+VwyRG0IgffJz0hADEOPqbQ95QeWZUky41yGX2bQuLQgtWmtqZYSleq7vqH8/OvgK69VYytEMD523zP0R771vnxCnia7XYeuVyOrzcurvpWkksXnbXR1sIL/JGCiLVCERYSklNglDmyaLPRdAtzf7htDZcQsf0ponEksVwpTLWxAn4skJHb101JqU5EXMiWiEZFWBqz7rPy3i9R9bgyoSmKo0d4YHwBCVe+Rm6D16TDILoJH0wJTxAxcdaAvJqAm5bojWcP9waXuX7TQHHCvpFLHldmD3RnpOd3uNff2YuRsWfgV8TUeE6TsaqGmER4jKfdw8qKNyyOCZ92bypQfhWDOPPuO/8fvbFoOA==", "tools/train_rmp_lm.py": "eNrFXP9z3ESW/33+ir5QV5LIWJ5xHG9wodQSSHJbtYEc8R1V53Op5FHPWGuNJCSNHZ/XW85CsEO++XYh7CawQBaWLEu+cLCQr+R/uY1m7J/yL9x73S2ppdE4gT3uUjAjtbpfv37v9ed9aY337NkzOPvF4C9n++98l3zy2eDMJnn12HGyffP69rX1/rWPkz+cfXx/8zVrib5MY7KXHDp++PH9MztX39y+cX1w741Hty/sbGz0P/p2+83L/fe/gK47H1xNtv6TvOi71jw5HvrkhWajQcbJzydh3N/Wf12rDc7cGfz6zuP7l2tNnSQ3ziWnr22/+eXgT/eSTy7tvHGN/Pznxx7fPze48nX/wqecKxL4vqsHK2Rw+U0ieIGpjloxtclL1I0tzlz/9NWd9cs7p34L18knl5Nbd5O7f4KlJFff2P725uCdL/tffgRcPL5/pTahk0cPHg7euUaOHp8Zm+Dr2oydxdhfpB7Mv+S3rHljf2NfY5INOUf9aDyigenY0Dqxf4psb3yevH0tef9a/9JG/+M/bn93Z/Duuf7t04M/302++3xw4ebg7W/666dwtn066f/1VHL/YvL2R9sPHpAjjkdfo/Njh+0ezBpZ3cClY83GoRmYCljfupBcvDW4+3Bw7SxIOrl4qX/mt8m508DEo9vnk/MfDf54anDld39bPwW3/bOXYNlMyjjTpE7m280pEEDbtcj2H84nW5vJW6f7H91G+Zw70//qKumElu1QLzZbC7S1GPiOFzteB57z7uQF2+q+hiK88GFyfat/6Tqoa3DvOmic6++da/2v3kX9ERKsxAu+R2LQTzQeh5bjmWE3MN0uaovdk7GxIKQR6KfL7/xeTMZbvhcDB+N26CzR8WMrL7FvHNrcd6ALvZgaIjLRSP89aTIuxCdRHnfBZqJYD2LGl9+Fiz2veC1KegGQtkjsdOme2p49e2pON/DDmFhhJ7DCiNbJLyLfq8Mq4oU68aM6eb1He9AcrcB1vBBSEKrXqTMKtTaQJgF0dZ15Iigdh9uUauyHreKN7nl6u+e1Ysf3LJdYETlSq736yiszxGAjVdNsOy41TU0HcfruElU1HTiDtYqvmtMmURyqOEgjnh8TkDdwpyMf0yA/kt3pjhfRMFYb9XyEVhNcg3hTlsVeq9UOvzJj/uwlYIWZfu1fX3nxhUPmiZ/922HWBHuEkGfYs5+Q7Y8/3XmwlWzeIlOTuGGT9fP9d2+B7Sabf3509wKZAb36IQBESEly8w70rdVqNm0Ts2stUtOmLWslUo/WiesbDX1/nSw48P3ccxMaXwPoZvDOh/3Nrf5fTycP/9L/9kuwf5hnZ30LjFvs9o9vJRsXB1/dYzNvwjRwMbh+ZvvbD4CF/nvf7Lz3dXL7NjSS2OrBGprjanPM0jhEofZxJpDnUWLAQz4x/gtp3As9MrvgzLE2GG26fp19LzhISEewU/FrDFag1YstC45Wk+mEfs+zxTPRkZMkz/IrIDouZtHIs9CI9+pR7K5pQH2/RtogTQd1HVpeh6pHtTkQ6PFXD584PHMCOFplEz5DHt2+l7x9BYAKBPIr2AvHSH/9XnLx1yiEcfKrqSY0PFzvfy3aGOhdZuA92Th6iCE4mZiEq/6Fa9vntxhVBfe0Mk1spxWrmZA4dOY2AkbGcZNbUZ3Q7rwxNQnG9xL7ygYeMybqxDb2TcCjLnyh7m2berzz6+xB1vmo0YQOLxvPTdXJIg4MxMiADw0KY0P4zsfGtBuAUTXrIDYnNpQowO2r1EnYc6mhtOKTcL0Mxm/CJg9XjJkQdzprgGf8NiMGLdxqjYIFI3fMfrnx1sk8jS3ocdKY0KVlzPtx7FIPgNg4YrmIM8iKCduSWl0xcexQE2XG7xC+TcDtuMSH64eWCTawaIiVanWh+u0H93Y2LgrVD2m++VyV5mFr7ax/mLz1m8EbH0KvmUn4ePTgAvijwY33dj4/t33zFDeBqGu57g+zAaaul4pqFTbQnCrbAOv8Onsg2cABNAFQbWYCbGDJBPjQENv+T03ggGQB/08GsCbAtQVAE5vgKKxupHZ9m7oCTmM/BndjkKjXVQPd63Wpq3JMCRBTWFedjaMxDSNV4wCGXsIMnjTOtVZgzGxzTm8B3FeQWUYSpb7sQTu0WvBIXQ70RQhF4CtaAC9nawh/cPcyfMPXUU7GAre5RKE/hDIqX9FYyuKzKfYiSa0Av6xnXYwWgprvOa5t+gG4cec/aMhlBc4oBOVTp7MQC0Uz00GNRoYK+mVK3q8JobIudfDBvDMwNjtXh//50kBGHsihLksKG2xTFlDudMAPoTcPwPG/3nPA+5toAflzZoE+BnE9Kg8CtdhOlzxPJghMqYABKTghTsUaPD/sFlrwW6eeHS07EHAo844VKRrrOm9axbHzJq693NaxigQVv4MOIm0qspxKR7eCACZVAy17TGEfFDtX9eyA9wwiFO6qwi0bYYiLXpGVBc3y7VqdSANSNobHNPTGGldZL6JmGz5smIxHa62ebelOZFpLlgOZjgvRWNGysBOzIp2F0ypnFg3JQFvilsM+AQuDyGjSMYALNomRTacJo+zQ2HRDNYppUOdWa+J1hOHoSZPZphV2ewFvhfwEwRPCYzcEPICIkltrBPG+DdDmR5B+KMLAeDfchYwSbBd5YBoHIV0wJHmWoaAoI8D4hE3bxM0KrWqzyJ+WUUWGMMJSliNbyQkyBUBfC0JQaU/z0TBBQz+wX5MtXbAnjStaT4HD7AnE/x1cuOMhQtQF32MyGWkFMgvFPtoISYwRNbviQoUQjs1a2236gqhGzV/oVDA9odC96HpQGRUsMEDcy1IZAGbwB3gROII5zARqLdeKInKCOSFIa45brUUaZgG4nKHyxLZ/9gzkqeSfep0OdD9iQUYFoUT//I2dK29B7J8l2STZfGv75vnkzjc8p4U4Prlxeefdh0Q9VCcze8Fm+mfu9L/4WIrDWXJgops2TTWibhsCBAs2DmwKjqPpHYBg24EUbN6KWwtmBPiNYcfrJrjXOlkC6fGs0oCs0mzgHokobIhJiBSiRScwbb8VGQ0ZeEV+JqoCWTvLk8SkUdrL9cEri8asJ3ILiIquLKWi42aGJh/TRVXpBPGEohUHHKrzb8z9KtZS7FwURaGJywOI7CauIjWUiKCSyQS9PG9OW4pjwAp99ONZEqwfXoJ8VC2tynwd+rCkWf9n/ETLxHUZB8Dg0p7PkOT0ZrJ1rn/90/7bD8Ckkit30UBYwMlsabP//npy5rykT0htPidp3ebRww+gZ7L1JtjQ9sOt7Y/PJRdv9m++A1Ftji+RucSCHlln6u42FQUuhoms7KCwtJnvDBaD5StFruZ7bW5uQnjo+xu58YBfDP1l9ImcjyJUxfQkgh70QENRFbyH+RTJRqSYAJ8Wx1cGAxJnOoxAH5paps5MkZp+CJqzwhUVSWpa5VDhfnkIP9yFLXcv5MBlTsFmVUFDIwcNSXXDvM+DXBeLhoPdHTvKfG/MKggpxdnpnBwI2o5XAmrwjq7vdbRhWkIt6WXJSrkVF8x5hl1BPh6CSgzebdkPARHRTmjX90pWIFPSmZNQhY0zLONjmQoktHl6VKGB31oADnOj8iiLTTh6MBfM8QOdcJ5p9NrMGrOG5QXHpcyO8n2MQQ3uBq0Uf0Xl7fJ04PPUG0eYilgarG3YNBgPdsSASS1qc28Jsop0h8PJ3clpkGz02m0I6piDyIARpmH81VGUbRoyWDaazJkUZyxs8uGpMWR5ksSLMcXQo6fEiSdixUi8EBbzQ/AiHboLXuTmh+CQAgOa8QgWF3reYhkB2O7HMVX7Xl9y6LJa8qeF/VBiF50tEGQ8zFV2+h77paCqcGX0Q8k96kEvVtlCeRXZ78VYAdV2HVvEy/I/erJFg1h43SM9192dkUpD4FuS4XoGYR4o3WShSRqQ0SWnRSUhnFxJ8QgWhhY6FCWfXJmdrpPpseacHvsqJ4A5s2fOu35rMYOIuujZnH5Cx5y9lutHtIyvks6YwiDK/akovfsso4YmHEwBBnpWTNPUX7ifEWFlyo7VBeRBG+Q5GetLIa2bSvMsluYjbSEJHs8Do0DZQzDHHIAjOmCkY9NC8Mcxnc2ZG3Gp8suTPjX1tsCzBmE/J1bgCojxVq2eXpQUJ8bPOtMOYivrM8d3VIUUGEe5bnJNYx1BbFmQj271Yr9lRYBWmD4r6aaVREc9zKVtQ1BiuSHrq5X2l+t3nBgkZ6b1IzU3J63UM0KgP6K3QriCyDsO/WBF5QT0Nvg0sAU8U1mwAqqONeuCto5LhHutYIB5PwnzclXiNuEk8U53YtpVpY5eHh5xe2BeURhEl1qeKbiVKGYJoFdM89Le/EgKYDpQUftS+wSYlDbSzDvUo5DjUxMBPLV1fiTGjcWjy2nC1MQCKlZMcUQvpABLBzAjDcxFY38j3QSGUBXXVFXy9LSZEI/2qrwNZ5AVpWa5X+FQvWxFJhMmnmQasnShQaTbdMmM2FnecM1R5w9MUQqs3LC7DgGSiEGYwPQvXBtsvdV//8Pk/nqydZO8oi5qZHBtfXDpYv/MZ5CQcJw6WfZms7DqCkeWSVfeXNmWyYjMM7NrTqG3H9o9xPLsUuEKT4nNqBegkijYhMaiJNFJ0Kr9eHs43b8QHMc038N8dXhEDNGmA7MgiGVYZ+ZYVzRQibAHjokTZwErXsyCheK+nmYHlHPpnhc7SzJr2OJ0rFhZYjZODpbD0SWOPEJ/frCo5vNyFGYD6xI7OZ4U8anAsNS9a0WL1MZDX1ciTp4nS7OAcXUBM8qY47UViSRskHmOd5HfjnGBMme20zXGpBDIO5lX2HAp3Z4LAYDfdcDqGaU6InsZxkA/aWwHBAryYgQNwrdmUWbFSOXHdw1F0yputVngc+4pNpugIkKLJyGABDFpiVPGpZzNYfAXqI6QZ1OEPHU2ZjbPTvFxQ6Axgva5DaeF4YidRy0GsRqw9xIEjPuI4a02Zl+iZIy7BOJUjFjmaRRj3UGIDQem7xjgtZa1pq8XdBdtJ1T5TSTOoOhJ2J2mvyhlcXEXi0BsICoXwKXddk6qig4PlDTmQUEj0+pqJgyFMa1MC6kwgZvsNFHLz7YUWBJ0gc9RHWC50AEXnbfh6qGRCSFvlaQBD2XZ5H1SIUGH9JI/XavjQgVORRAPBK7Voiq01QkXnzhw69pc8aoVdqLsXIibbgrLDGBGHCdwPFZaQU9Jjals+dNSBMJ8AFZpPUhpGTkIC2A765br+oCT7X0TwkftMsjzRnTHEwn2XpEx7Fy0p/AuT+G0silG+KGoZbk0zIYiaByFaOYEa85gYyRKMCYxdUvn0cSmBovBOgCak3hpYhY1pvO3luaywwrWCCEU7rZc8DB6VskOZJU5RF48xc3xAlrEazTqs89C96FAmcd6QXoOyU5VK45rRV9mrAHWHvC0wOBssZgconp2M5NeMLasVqvXlebhBwhiHL9JF8fvwNdxFYiwkz0Shdbx8TIHQogAULAjwxRH2Bho0+SHT4sjsMdZKl55DssoDx3GstZlWzDDCm7mLsiH2VYd/xM+lLlQNpS/HmYKTExXNU6U7L0xpWAPYCO9LmW2JQ3V2aoKZ7itvHiBVTRV6o3RdgAeuoUHbp6RJpR8gZHpe+4KfzNAK/kPVo6T0LC1OCuwVEqBEDGrOiKeSt1ymYl4APsw+JyT85xMmFIvGU6lzpKwuZhbi7xSlUFrUf6FGAZpt5XZV5l458ijexfIqiSyNdI/dTX55Pzj+5eRR2M1Z38t1bmxKjE23qRT03qzvXYstwJjNb2a1ifba4qwnk7QY+XM4lEvpipcM+yh2tCqAFlg9ovH/0WpFRZyDNUyRzimGKsSwKyRX/KdifwyIOC8TgCv8IhjAowQ2FB4KGKV1ZRlpMUjmtUMa9eEZWeszKBTmuOlBcEJu16DjGRGNMzgDUMO0ZBDyRq+UlYCgen6Ggp9nJnPL2WgSRfFT4yhmyo3PFuiA4t7ji3uEJLTUo08gy+uJt/+1/Znp7huyaP7twY33tv+9qvthxv87Vvxot+pje0b3yV3vko+2OjfvDi4uUEONBrCIPhrvDunHianz/PXWXc2zoMRYZr2ydfbN6/3L/2e44d08MVD49zsAQGBosYPnbMtk2FmQwRPeFoKo0vnpxwXRYFcYFnprA4ivpd9j9YlXBfXM+K4kkN18TzOKBW+M2W/BNTnyKo8E+qgYuY1jSS3vhzc+xxllJ62waajnsqXkx7DaELb6A/UVekZTr9GGBOp5uLsEB/rmTp+pNUvTEkq2rFygYbGC2IVlZLaUEUVo2SmhSw5LDqAzNzKeQJqaPSbFbmnkV+t4J6GtaQvVHCF4OXwGUDQYeF7EPN3n0z+FshwETaAKMINWfjghrXCYxwMDlDUblBhsW96YCMVByjIY1pCSsUnsyMl0KVtXVG+PlknKyygZxqWKr7lKt+Pn9LtUvXTRnT8+4t+K9XFvoppRJ2uKugqn/qwaBUjZLH2UecGoiP7YrVEjQXqYHm2OsxL9QFXyuFuI3OLqa5c1n7IAkSnnscuTBVMeHhmJih8BS0LELL34Hux40I+4gKmZb1MdfhVRPZydYVqUuHB2qrnThkMbKzzP81h4Y/ILc9pgdWStGW3s9coB95DNW6GmtAv0+iQ7qT3sv5RwJvfMekSDVfYiStJgdQwJOdWcXAa89fF1CYdm6rLEE7GOLAPL9LLEo9m9sJWYRruP4tcDR96+osm0lC97F2wcj4C+9COhwZaSx1T2qxMVuMpU8MLpAGG5Ta1bAlr5MIPxNJRxW7qUoD4+WLkiJVKaPfDFRNz65bFEmKYHYKd0TFkGb1zr165zdssVier+Lm3OX1wyl4blyMtHmUukqGY+OBPWFQMj5URhJnYVlMBslCZqEHgktXCuUPaAY8dAL6RqrYb1RBiixCCPRa0djyymu0dFgFC4yrT9nRdb7RFeLkLPdQVWcXPdPgS7DzgkWmEsXP00PDgYRVWBCAFC6yIW568z/AUobTRpE6GIUcew/ttKTsnxCsUvTF0SFmM0arDx6GjyipHYgURtU3MftXyxs7eiNw31RhlnW3l4MGDZPYwJn4/JbJRrs1lCzFW0ytuTr9MF8YfwEWqRMGPsZoxxp4sVLzlIH4AZlQfbCkzCxS2cRSLcynLi4ntRC2fKQV/KNVyKP4OzImUoeOvA43KQ5iRqz/BWIG4m/P0D+Fa9VsZqRTI81lSWu3FpVw6HVMdNWTlYKl0MS8KFyOqw2iDoyrE1SGeH0BSIE0ADi50WpGOP5DDrF7hhTh+rmcovbg9dgCrhBFpVy+vrS+HEHCoSEC3e90gksrD5X9yUZfzLioQpVouUVJRwQP+O6t8I01KleOhCYQNFobBPSAb+HAls8TsedZSJ/tG0AUs3EuUf/eUp0ELpsUfjBa5DRRrTH+37rOk8gTMMEcgIX/08IPk+u/4D0TJf7/1m2KNRth82/HA70n5moAq/o6GXC7nu+V/q15errgx0aJcvme9raKIuzjLThvmhkq5T1mXE+9uVQKVKA1VHsOzZ9jG74tH8rxOS7tBejAvCrfsGLICvbg62dtbQgddlmGLUyFeImZnWfoLYafXhRzqON6FIu6JehjoQJpnw2J786wnhLk26N9QQJ34Uyr+exVbfj0nDtmPhubZOD5IFS8HCuGE7JEl5lSV9AfEQLC14AP/kcGOhkW1XsPVta2eC9N2dyUDkFVmqk4WqBsYyvZ3v0027j66fUH82vnKjeTBu4/vb/KfsT+6/Xb/3Ab+rP2o73fAybAfFT++f2b0XKK4okjMSe/GH3ltHLYFXabzY9TuPZGKKNHIxKTfjI8ezvw/jGJ5N2g7H9+cGjlopnpAY2Jy5JA8560cO5lKub9+T7zSJOpH4g0nCOPx0EIiM3IqAfVV00zga5jp/+mUoMn+Rxv8jxWI4iFwwauFvKLTIP33vsFfFL+7uatGWe/KibPJts9+ykqByebv8af21z+FeZJT7/evXx1N1g1TmiytzKlO0bHREl+2R4xq6M3Rg1jZqnIJ+9MXWKsWjqUtafPNpr8gqvPf7cxJZon3o5ea5nXV6hvNQR4+f3/mM1/6/YdSalfb88TIMfw8SOFneeBcIFjxQwB4gJrRchGHitWjUttKTn+189710l9wIIW/4JAGF1FQAbEcMAQTUVBigs0/4ln6hxJk+Cn/yYTRY3N3NULrjZFD0ZmNNPMDo4ehyxuhbSEhdI3MeYkXOOBWLR70ggNjKTn3TNI5XvFYvzZcMSpHMuBcHfz9Eh7QmCYjaproqExT0OV+t/Y/yy9+zQ=="}
DST = Path("/content/rmp")
for rel, b64 in BUNDLE.items():
    p = DST / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(zlib.decompress(base64.b64decode(b64)))
    print("Wrote:", p, p.stat().st_size, "bytes")
%cd /content/rmp
!python -c "import dspn_text; print('USE_FLA =', dspn_text.USE_FLA)"


In [ ]:
# 4. 2 分钟快速冒烟测试（确认 A100 吞吐、显存和 loss 正常下降）
!python tools/train_rmp_lm.py train \
  --preset main \
  --out /content/drive/MyDrive/rmp_138m_smoke \
  --batch 16 --T 1024 --grad_accum 4 \
  --steps 20 --log_every 5 --eval_every 20 --save_every 20

In [ ]:
# 5. 正式训练主力档（~138M，目标 2B tokens；自带 --resume 断连自动续训）
# A100 40GB 可尝试 --batch 32 --grad_accum 2（同样 65,536 tok/step，吞吐更高）；L4 24GB 保持 --batch 16 --grad_accum 4
!python tools/train_rmp_lm.py train \
  --preset main \
  --out /content/drive/MyDrive/rmp_138m \
  --batch 16 --T 1024 --grad_accum 4 \
  --tokens 2000000000 \
  --lr 6e-4 --warmup 500 --sched wsd \
  --log_every 20 --eval_every 500 --save_every 500 \
  --resume

In [ ]:
# 6. 随时测试模型生成效果（自动开启推理期 O(k) 稀疏执行）
!python tools/train_rmp_lm.py sample /content/drive/MyDrive/rmp_138m/best.pt \
  --prompt "In the near future, artificial intelligence will" \
  --max_tokens 200 --temp 0.8 --top_k 50